In [45]:
from astropy.io import fits
import numpy as np
from astropy.table import Table
import pandas as pd

In [ ]:
hdul = fits.open("CIV-Absorbers-dr1-v1.0.fits") # lya quasar catalog
catalog_absorbers = hdul[1].data
catalog_absorbers = catalog_absorbers['Z_ABS']
catalog_IDs = hdul[2].data['TARGETID'].astype(str)  # Ensure TARGETID is string type
catalog_z = hdul[2].data['Z_QSO']
redshift_dict = {tid: z for tid, z in zip(catalog_IDs, catalog_z)}


hdul = fits.open("CIV-Absorbers-parent-QSO-dr1-v1.0.fits")
parent_IDs = hdul[1].data['TARGETID'].astype(str)

hdul = fits.open("QSO_catalog.fits")
analyzed_IDs  = hdul[1].data['TARGETID'].astype(str)
analyzed_IDs = analyzed_IDs[0:99] # remove for differnet number of quasars

df = pd.read_csv("CIV_catalog-validationtest2.csv", dtype={0:str})
detected_IDs = df.iloc[:,0]
detected_absorbers = df.iloc[:,[1,2,3,4,5,6,7]]

In [53]:
# print(catalog_IDs)
# print(detected_IDs)
# print(catalog_absorbers)
# print(detected_absorbers)
print(type(detected_IDs[1]))
print(detected_IDs)

<class 'str'>
0     39627797555054963
1     39627797563442938
2     39627797563446432
3     39627797563446682
4     39627797563447018
            ...        
65    39627827766627476
66    39627827766629916
67    39627827770818892
68    39627833798035798
69    39627833798036300
Name: 39627791519451554, Length: 70, dtype: object


In [ ]:
dv = 350 # km/s, velocity difference threshold
# Convert dv to redshift difference
c = 299792.458  # speed of light in km/s
z_trh = dv / c

TrueP = 0
TrueP2 = 0
FalseP = 0
FalseN = 0
total_catalog_abs = 0
total_detected_abs = 0

catalog_absorbers = np.array(catalog_absorbers, dtype=float)
detected_absorbers = np.array(detected_absorbers, dtype=float)

surveyed_IDs = set(set(parent_IDs) & set(analyzed_IDs))  # IDs that are in both parent and analyzed catalogs
catalog_surveyed = set(catalog_IDs) & surveyed_IDs  # IDs that are in both catalog and surveyed IDs


# Create a dictionary mapping TARGETID to Z_ABS
catalog_dict = {tid: zabs for tid, zabs in zip(catalog_IDs, catalog_absorbers)}
detected_dict = {tid: zabs for tid, zabs in zip(detected_IDs, detected_absorbers)}

for tid in catalog_surveyed:
    arr = np.array(catalog_dict[tid], ndmin=1)
    arr = arr[~np.isnan(arr)]
    total_catalog_abs += len(arr)

for tid in detected_IDs:
    arr = np.array(detected_dict[tid], ndmin=1)
    arr = arr[~np.isnan(arr)]
    total_detected_abs += len(arr)


for i in range(len(detected_IDs)):  #loops over all detected IDs
    tid = detected_IDs[i]           #sets working ID
    if tid in catalog_dict:         #proceeds if the ID matches to catalog
        catalog_zabs = np.array(catalog_dict[tid],ndmin=1)   #their absorbers     #Create arrays for both sets of absorbers
        catalog_zabs = catalog_zabs[~np.isnan(catalog_zabs)]  # Remove nan values
        detected_zabs = np.array(detected_dict[tid],ndmin=1)  #our absorbers
        detected_zabs = detected_zabs[~np.isnan(detected_zabs)]  # Remove nan values
        dz = np.zeros(np.size(catalog_zabs))      # Initialize dz array
        bool_match = np.zeros(np.size(catalog_zabs), dtype=bool)
        for current_z in detected_zabs:     #Searching thru our detected absorbers
            for k in range(np.size(catalog_zabs)):
                dz[k] = catalog_zabs[k] - current_z
                bool_match[k] = np.abs(dz[k]) < z_trh * (1+current_z)  # Use redshift of the quasar to scale the threshold
            if sum(bool_match) > 0:             #Absorbers match, true positive
                TrueP += 1
            elif sum(bool_match) == 0:          #Absorbers do not match, false positive, we found it but they didn't
                FalseP += 1
        for current_z in catalog_zabs:      #Searching thru the catalog's absorbers
            dz = detected_zabs - current_z
            bool_match = np.abs(dz) < z_trh * (1+current_z)
            if sum(bool_match) > 0:             #Absorbers match, true positive (redundant)   
                TrueP2 += 1
            elif sum(bool_match) == 0:          #Absorbers do not match, false negative, they found it but we didn't
                FalseN += 1
    else:
        detected_zabs = np.array(detected_dict[tid],ndmin=1)
        detected_zabs = detected_zabs[~np.isnan(detected_zabs)]
        FalseP += len(detected_zabs)

print("True Positive:", TrueP,', ', TrueP2)
print("False Positive:", FalseP)
print("False Negative:", FalseN)
print("Total catalog absorbers:" , total_catalog_abs)
print("Total detected absorbers:", total_detected_abs)
print(" ")

purity = TrueP / total_detected_abs
completeness = TrueP / total_catalog_abs
print("Purity: ", purity*100, '%')
print("Completeness:", completeness*100, '%')

True Positive: 27 ,  27
False Positive: 168
False Negative: 4
Total catalog absorbers: 33
Total detected absorbers: 195
 
Purity:  13.846153846153847 %
Completeness: 81.81818181818183 %


In [49]:
print(TrueP+FalseP)
print(TrueP+FalseN)
print(len(surveyed_IDs))

195
31
1000


In [50]:
#Detected IDs not in catalog
detected_only_ids = [tid for tid in detected_dict if tid not in parent_IDs]
print(detected_only_ids)
print(len(detected_only_ids), "detected IDs not in catalog")

[]
0 detected IDs not in catalog


In [51]:
#Common IDs
common_ids = list(set(catalog_dict.keys()) & set(detected_dict.keys()))
print(common_ids)
print(len(common_ids), "common IDs between catalog absorbers and detected absorbers")

['39627833768674499', '39627857927864461', '39627845844080517', '39627815695422771', '39627821731024730', '39627803615825492', '39627870015852399', '39627815670256963', '39627797563447018', '39627815666064157', '39627809655624495', '39627797555054963', '39627851883879796', '39627797563446432', '39627821718439126', '39627827770818892', '39627839808475565', '39627803603242846', '39627809634648198', '39627809634651155', '39627815670259013', '39627833798036300', '39627809638848413', '39627906204305027', '39627815678646727', '39627888114273647', '39627809647236885', '39627821731024622', '39627821718441004', '39627863971861566']
30 common IDs between catalog absorbers and detected absorbers
